# ssl - settling the constants for a self-relative TLS-setup detector

> **Finding:** see the summary cell at the end - every candidate constant is stated with the population it was measured on and the per-week finding volume it implies.

## Inputs — bring your own logs

This notebook loads Zeek `ssl.log` / `x509.log` through sigwood's own loader: point
`SSL_SOURCE` (and optionally `X509_SOURCE`) in the first code cell at your Zeek log
directory or files — rotation and gzip are handled, and the frames are the same canonical
schema the shipped `ssl` detector reads. Run from a clone with sigwood installed
(`pip install -e .`). Every output is an **aggregate** — counts, shares, quantiles, sweep
tables, and TLS parameter names (cipher / curve / ALPN strings are protocol vocabulary, not
network identifiers). No address, hostname, SNI, certificate subject, issuer, or fingerprint
is printed anywhere. Direction derives from the `HOME_NETS` setting (RFC 1918 default).

**Provenance of the quoted numbers.** The prose here — including the settled-constants
summary at the end — quotes the calibration run on the maintainer's own 19-week archive;
those figures are the record behind the shipped detector's constants. Re-running prints
YOUR estate's numbers and moves nothing that shipped. Two fields explored below
(`sni_matches_cert`; certificate `sig_alg` / `ca`) sit outside sigwood's canonical schema —
their cells say so and degrade when the columns are absent.

## Question

*Does an outbound TLS session's setup look unlike this estate's own norm?* Self-relative, list-free, no JA3. Four candidate legs, each assessed against its real denominator:

1. **tuple rarity** - `(version, cipher, curve, alpn)` rarity within the window (TLS 1.3-proof)
2. **SNI-less established outbound TLS** (1.3-proof)
3. **validation class** on outbound flows whose certificate was visible (≤ 1.2 only)
4. **x509 join** - validity shape, cert age at first use, SNI-vs-SAN

A leg ships only if its population exists on this estate and its per-week finding volume is one an operator can read.

In [ ]:
import ipaddress
from pathlib import Path
import numpy as np, pandas as pd
from sigwood.common import loader

pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

# --- point these at YOUR logs ---------------------------------------------
# A Zeek log directory (flat or dated - rotation and .gz are handled), or a
# single ssl.log / x509.log file. x509 is optional; the cert-age cells degrade
# without it.
SSL_SOURCE  = Path('/var/log/zeek')
X509_SOURCE = SSL_SOURCE
# Networks that count as "home" for the outbound gate (RFC 1918 default).
HOME_NETS = ('10.0.0.0/8', '172.16.0.0/12', '192.168.0.0/16')
# ---------------------------------------------------------------------------

def _load(source, pattern):
    source = Path(source).expanduser()
    files = [source] if source.is_file() else None
    return loader.load_logs(source, pattern, _files=files, _warnings=[])

ssl = _load(SSL_SOURCE, 'ssl*.log*')
x509 = _load(X509_SOURCE, 'x509*.log*')
if ssl.empty:
    raise RuntimeError('no ssl.log rows loaded - check SSL_SOURCE')

# The loader returns sigwood's canonical schema; this notebook's column
# vocabulary predates it, so rename once and fill optional gaps with NA.
ssl = ssl.rename(columns={'port': 'dst_p'})
for c in ('version','cipher','curve','alpn','sni','resumed','established',
          'validation_status','cert_fp','dst_p'):
    if c not in ssl: ssl[c] = pd.NA
for c in ('resumed','established'):
    ssl[c] = ssl[c].eq(True)   # strict-True, the shipped detector's own posture
x509 = x509.rename(columns={'not_valid_before': 'nvb', 'not_valid_after': 'nva'})
for c in ('fingerprint','nvb','nva','self_signed','key_alg','key_length'):
    if c not in x509: x509[c] = pd.NA

HOME = [ipaddress.ip_network(n) for n in HOME_NETS]
def zone(v):
    try: a = ipaddress.ip_address(v)
    except (ValueError, TypeError): return 'bad'
    if a.is_multicast or a.is_link_local or a.is_loopback or a.is_unspecified: return 'nonroute'
    return 'home' if any(a in n for n in HOME) else 'ext'
for col in ('src','dst'):
    z = {v: zone(v) for v in ssl[col].dropna().unique()}
    ssl[col+'_zone'] = ssl[col].map(z).astype('category')
ssl['outbound'] = (ssl.src_zone=='home') & (ssl.dst_zone=='ext')
stamp = pd.to_datetime(ssl.ts, unit='s', utc=True)
ssl['week'] = stamp.dt.strftime('%G-W%V'); ssl['day'] = stamp.dt.date
ssl['cert_visible'] = ssl.cert_fp.notna()
ssl['tuple'] = (ssl.version.astype(str)+'|'+ssl.cipher.astype(str)+'|'+ssl.curve.astype(str)+'|'+ssl.alpn.astype(str)).astype('category')
def pairs_per_week(df):
    """Distinct (src,dst) pairs per week, zero-filled - safe on empty selections."""
    if df.empty: return pd.Series(0, index=weeks, dtype=int)
    s = df.groupby('week').apply(lambda g: g[['src','dst']].drop_duplicates().shape[0], include_groups=False)
    return s.reindex(weeks).fillna(0).astype(int)
weeks = sorted(ssl.week.unique()); full_weeks = [w for w in weeks if (ssl.week==w).sum() > 0.5*ssl.groupby('week').size().median()]
print(f'rows {len(ssl):,}  days {ssl.day.nunique()}  ISO weeks {len(weeks)} (full-ish: {len(full_weeks)})  x509 rows {len(x509):,}')
print('direction mix:'); print(pd.crosstab(ssl.src_zone, ssl.dst_zone))

## 1. Population - what the cert legs can and cannot see

In [ ]:
print(pd.crosstab(ssl.version, ssl.cert_visible, margins=True))
print()
print('cert-visible share by version:'); print(ssl.groupby('version', observed=True).cert_visible.mean().round(3))
print()
ob = ssl[ssl.outbound]
print(f'outbound rows {len(ob):,} ({len(ob)/len(ssl):.1%})   outbound cert-visible {ob.cert_visible.mean():.1%}')
print('outbound per week (median / min / max):', int(ob.groupby('week').size().median()), ob.groupby('week').size().min(), ob.groupby('week').size().max())
print('outbound distinct (src,dst) pairs per week, median:', int(pairs_per_week(ob).median()))
print()
print('dst port mix, outbound:'); print(ob.dst_p.value_counts().head(8))

## 2. Leg 1 - tuple rarity

Rarity of the `(version, cipher, curve, alpn)` tuple *within the week*. The estate's tuple vocabulary is small; the question is whether rare tuples are stable noise or a usable signal, and what bar yields a readable volume.

In [ ]:
tw = ssl.groupby(['week','tuple'], observed=True).size().rename('n').reset_index()
tw['share'] = tw.n / tw.groupby('week').n.transform('sum')
print('distinct tuples per week:'); print(tw.groupby('week').size().describe()[['min','50%','max']])
seen=set(); new_per_week={}
for w in weeks:
    t=set(tw[tw.week==w].tuple); new_per_week[w]=len(t-seen); seen|=t
print('new-to-archive tuples per week (after week 1):', list(new_per_week.values())[1:])
print()
print('overall tuple table (top 12 by rows, then the rare tail) - protocol vocabulary only:')
tt = ssl.groupby('tuple', observed=True).size().sort_values(ascending=False)
print(tt.head(12).to_string()); print('...'); print(tt[tt<=50].to_string())
print()
print('rarity sweep - outbound (src,dst) pairs per week that used a tuple below the bar:')
obw = ob.merge(tw, on=['week','tuple'])
rows=[]
for bar in (1e-3, 1e-4, 1e-5, 1e-6):
    f = obw[obw.share < bar]
    per = pairs_per_week(f)
    rows.append((bar, int(per.median()), int(per.max()), f.tuple.nunique(), f.src.nunique()))
print(pd.DataFrame(rows, columns=['share<','pairs/wk median','pairs/wk max','tuples hit','distinct src']).to_string(index=False))
print()
print('the rare tuples themselves (overall share < 1e-4), with their dominant dst port:')
rare = tt[tt/len(ssl) < 1e-4].index
print(ssl[ssl.tuple.isin(rare)].groupby('tuple', observed=True).agg(rows=('ts','size'), outbound=('outbound','mean'), top_port=('dst_p', lambda s: s.mode().iat[0])).to_string())

## 3. Leg 2 - SNI-less established outbound TLS

23% of rows carry no SNI. Split it before calling it a leg.

In [ ]:
nosni = ssl[ssl.sni.isna()]
print(f'sni-less rows {len(nosni):,} ({len(nosni)/len(ssl):.1%})')
print(pd.crosstab([nosni.outbound, nosni.resumed], nosni.established, margins=True))
print()
cand = nosni[nosni.outbound & ~nosni.resumed & nosni.established]
print(f'candidate: outbound & !resumed & established & no SNI → rows {len(cand):,}  cert-visible {cand.cert_visible.mean():.1%}')
print('version mix:', cand.version.value_counts().to_dict())
print('dst port mix:', cand.dst_p.value_counts().head(6).to_dict())
print('alpn mix:', cand.alpn.value_counts(dropna=False).head(5).to_dict())
pw = pairs_per_week(cand)
print('distinct outbound pairs per week:', pw.astype(int).tolist())
print('distinct dst overall:', cand.dst.nunique(), '  distinct src:', cand.src.nunique())
print('pairs seen in ≥ half the weeks (persistent, i.e. benign-recurring):', int((cand.groupby(['src','dst'], observed=True).week.nunique() >= len(weeks)/2).sum()),
      'of', cand.groupby(['src','dst'], observed=True).ngroups)

## 4. Leg 3 - validation class on outbound, certificate-visible flows

In [ ]:
cv = ssl[ssl.cert_visible]
print(pd.crosstab(cv.validation_status, cv.outbound, margins=True))
print()
bad = cv[cv.outbound & (cv.validation_status.astype(str) != 'ok')]
print(f'outbound cert-visible non-ok rows {len(bad):,}  distinct dst {bad.dst.nunique()}  distinct cert_fp {bad.cert_fp.nunique()}  distinct src {bad.src.nunique()}')
pw = pairs_per_week(bad)
print('distinct outbound non-ok pairs per week:', pw.astype(int).tolist())
per = bad.groupby(['src','dst'], observed=True).week.nunique()
print('of those pairs, seen in ≥ half the weeks:', int((per >= len(weeks)/2).sum()), 'of', len(per), ' - seen exactly one week:', int((per==1).sum()))
print('dst port mix:', bad.dst_p.value_counts().head(5).to_dict())
print()
if 'sni_matches_cert' in cv:
    print('sni_matches_cert on outbound cert-visible rows:'); print(cv[cv.outbound].sni_matches_cert.value_counts(dropna=False))
else:
    print("sni_matches_cert: outside sigwood's canonical ssl schema (calibration estate measured it dead: 2 rows in 7.4M)")

## 5. Leg 4 - x509 join: validity shape and cert age at first use

In [ ]:
x = x509.drop_duplicates('fingerprint').copy()
x['valid_days'] = (x.nva - x.nvb)/86400
ca_n = int(x.ca.eq(True).sum()) if 'ca' in x else "n/a (outside sigwood's canonical x509 schema)"
print('x509 distinct certs', len(x), ' self-signed', int(x.self_signed.eq(True).sum()), ' CA certs', ca_n)
print('validity days quantiles:'); print(x.valid_days.quantile([.01,.05,.1,.25,.5,.75,.9,.95,.99]).round(0).to_string())
print('validity ≤ 7d:', int((x.valid_days<=7).sum()), ' ≤ 30d:', int((x.valid_days<=30).sum()), ' > 398d (CA/B max):', int((x.valid_days>398).sum()))
print('key length mix:', x.key_length.value_counts().to_dict())
print('sig alg mix:', x.sig_alg.value_counts().head(5).to_dict() if 'sig_alg' in x else "n/a (outside sigwood's canonical x509 schema)")
first = ssl[ssl.cert_fp.notna()].groupby('cert_fp').ts.min().rename('first_use')
j = x.set_index('fingerprint').join(first, how='inner')
j['age_days'] = (j.first_use - j.nvb)/86400
print()
print(f'certs with a first-use in ssl: {len(j)}')
print('cert age at first use (days) quantiles:'); print(j.age_days.quantile([.01,.05,.1,.25,.5,.75,.9]).round(1).to_string())
print('age < 1 day:', int((j.age_days<1).sum()), ' < 7 days:', int((j.age_days<7).sum()), ' negative (clock/not-yet-valid):', int((j.age_days<0).sum()))
ob_fp = set(ssl[ssl.outbound].cert_fp.dropna().unique())
jo = j[j.index.isin(ob_fp)]
print(f'outbound-seen certs: {len(jo)}   of which age<7d: {int((jo.age_days<7).sum())}   validity≤30d: {int((jo.valid_days<=30).sum())}   self-signed: {int(jo.self_signed.sum())}')

## 6. Cross-leg overlap - do the legs point at the same pairs?

In [ ]:
def pairs(df): return set(map(tuple, df[['src','dst']].astype(str).drop_duplicates().values))
L1 = pairs(obw[obw.share < 1e-4]); L2 = pairs(cand); L3 = pairs(bad)
young = set(jo[jo.age_days<7].index) | set(jo[jo.valid_days<=30].index)
L4 = pairs(ssl[ssl.outbound & ssl.cert_fp.isin(young)])
import itertools
names = {'L1 tuple':L1,'L2 nosni':L2,'L3 validation':L3,'L4 x509':L4}
print({k: len(v) for k,v in names.items()})
for a,b in itertools.combinations(names,2): print(f'{a} ∩ {b}: {len(names[a]&names[b])}')
allp = set().union(*names.values()); from collections import Counter
cnt = Counter(p for v in names.values() for p in v)
print('pairs flagged by ≥2 legs:', sum(1 for p in allp if cnt[p]>=2), 'of', len(allp))

## 8. The two surviving legs - volume, per-pair mass, overlap, persistence

Legs 1 and 4 are refuted as *findings* above (legacy cipher negotiation and Let's Encrypt rotation are the rare tail). This cell sizes the two that survive, to settle the floor and the ladder.

In [ ]:
A = cand.copy(); A['leg']='A_nosni'
B = bad.copy();  B['leg']='B_validation'
both = pd.concat([A,B])
if both.empty:
    print('no leg-A or leg-B candidate pairs on this input - nothing to size')
else:
    pairs_wk = both.groupby(['week','src','dst'], observed=True).leg.agg(lambda s: '+'.join(sorted(set(s)))).reset_index()
    tab = pairs_wk.groupby(['week','leg']).size().unstack(fill_value=0).reindex(weeks).fillna(0).astype(int)
    tab['total'] = tab.sum(axis=1); print(tab.to_string())
    print('\nper-week total: median', int(tab.total.median()), 'max', int(tab.total.max()))
    pc = both.groupby(['src','dst','leg'], observed=True).size().rename('conns').reset_index()
    print('\nconnections per (pair, leg) over the archive - quantiles:'); print(pc.groupby('leg').conns.quantile([.1,.25,.5,.75,.9]).unstack().round(0).to_string())
    print('pairs with exactly 1 connection:', int((pc.conns==1).sum()), 'of', len(pc))
    for floor in (1,2,3,5):
        kept = pc[pc.conns>=floor]; print(f'min_connections={floor}: pairs kept {len(kept)} of {len(pc)}')
    pw = both.groupby(['src','dst'], observed=True).week.nunique()
    print('\npair persistence (weeks seen):', pw.value_counts().sort_index().to_dict())
    print('pairs seen in ≥ 4 weeks (allowlist candidates):', int((pw>=4).sum()), ' - their share of all flagged rows:', round(both.set_index(['src','dst']).index.isin(pw[pw>=4].index).mean(),3))

> **These figures are the calibration estate's** (19-week archive, 2026-08) — the record
> behind the shipped `ssl` detector's constants. A re-run on your logs prints your numbers
> and moves nothing that shipped.

## 7. Candidate constants - settled on this run

**Grain and gate (structural, not tunable).** One finding per `(src, dst)` pair per window, the exfil grain. Outbound gate: `src` in `home_net`, `dst` outside it and routable - this gate alone removes the home-lab self-signed mass (318k of 373k self-signed rows are internal) and is what makes the validation leg usable at all.

**Leg A - SNI-less outbound TLS:** `established and not resumed and server_name absent`. Population 27k rows / 33 pairs over 19 weeks; ~4 pairs/week, rising to 13 in the newest week. No threshold - the shape is the fact. 1.3-proof in principle, though on this estate 99% of the population is TLS 1.2 (SNI-less 1.3 is 153 rows total).

**Leg B - validation class on certificate-visible outbound flows:** `cert_chain_fps present and validation_status != "ok"`. 54k rows / 27 pairs; 6-12 pairs/week. Certificates are visible on 0% of TLS 1.3 rows, so this leg sees the ≤1.2 population only (27% of outbound rows) and the finding MUST say so.

**Severity ladder (dns shape):** one leg → LOW; both legs on the same pair → MEDIUM (3-5 pairs/week here; the two legs are independent categories - client behavior vs. server infrastructure). HIGH stays reserved for the cross-detector corroborator seam.

**`min_connections = 1`.** 22 of 60 (pair, leg) cells are a single connection; a floor of 2 drops 37% of pairs to save nothing the operator needs saved - the combined volume is already a median 8 pairs/week. Recorded as the alternative, not adopted.

**Persistence is the allowlist's job, not a constant.** 8 pairs recur in ≥ 4 weeks and carry 67% of all flagged rows; the existing numeric `connections` allowlist (a `dst` rule) retires them in one line each. No decay/baseline mechanism - sigwood is batch and stateless.

**Refuted as findings, retained as evidence:** tuple rarity (the rare tail is legacy TLS 1.2 cipher negotiation with old internet hosts, 564 pairs at a 1e-4 bar, vocabulary still growing 0-7 new tuples/week - a fact about old servers, not attacker shape); cert validity shape and age at first use (Let's Encrypt makes 84-90 day / under-7-day-old certs the *norm*: 16% of certs are under a week old at first use; 420 pairs flagged); SNI-vs-SAN mismatch (population: 2 rows in 7.4M - dead). Version floor (TLS 1.0: 15 rows) rides evidence.

**Evidence per finding:** `severity_basis` (`["sni_absent"]` / `["validation"]` / both), `validation_status`, `tls_version`, `tuple` + its window share, `conn_count`, `first_seen`/`last_seen`, and - via the `cert_chain_fps[0] → x509.fingerprint` join when present - `cert_validity_days`, `cert_key_alg`, `cert_self_signed`. TLS 1.3 blindness is disclosed by a runner note carrying the cert-visible share, never implied by a finding.

**Open for the planning session:** whether the detector takes its own seat or ships as a second evidence edge for the corroborator; and the W31-W34 rise in leg A (1 → 13 pairs/week) is a real observation worth an identity-bearing look outside this notebook.